In [ ]:
import numpy as np
import h5py
import pickle
import copy

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
rundirs = [
    'goamazon_2pulse.largedom.r20251008.rerun',
    'goamazon_2pulse.largedom.ehe1.r20251030.rerun',
]
casenames = [
    'CTL',
    'EHEall',
]
casecolors = [
    'black',
    'green',
]
stats = []
for rundir in rundirs:
    with open(f'{rundir}/pkl/csd_stats.sparse.claude.v2.pkl', 'rb') as f:
        stats.append(pickle.load(f))

In [ ]:
minmf = 0
maxmf = np.max([np.max(s[0]) for s in stats]) + 10.0
maxmf = np.log10(maxmf)
print(minmf, maxmf)

In [ ]:
nx, ny, nz, nt = 512, 512, 100, 241
dts = 0.5  # minute
dx = 250   # m
dy = 250   # m
dz = 50    # m
grid_vol = dx*1.0e-3 * dy*1.0e-3 * dz*1.0e-3  # km**3
z  = np.arange(dz/2, nz*dz, dz)          # (130,) cell centres in m
t  = np.arange(0, nt) * dts              # minutes
ti = np.arange(-0.5, nt, 1.) * dts
zi = np.arange(0., nz*dz + 1., dz)       # (131,) cell edges in m

In [ ]:
def compute_cloud_profiles(csd_stats, rundir, CHUNK=2000):
    """Read HDF5 and compute per-cloud time-summed wq and qc profiles.

    For each attached cloud j:
        per_cloud_wq[j] = sum_t( wq[j, t, :] ) * factor   shape (NZ,)
        per_cloud_qc[j] = sum_t( qc[j, t, :] ) * factor   shape (NZ,)

    Results are cached at {rundir}/pkl/cloud_profiles_wq_qc.npz so the
    expensive HDF5 reads only happen once.  Binning is done separately.

    Returns
    -------
    per_cloud_wq, per_cloud_qc : float32 arrays of shape (n_attached, NZ)
    """
    global nx, ny, nz, nt

    cache_path = f'{rundir}/pkl/cloud_profiles_wq_qc.npz'
    try:
        cache = np.load(cache_path)
        print(f'Loaded cache: {cache_path}  '
              f'shape={cache["per_cloud_wq"].shape}')
        return cache['per_cloud_wq'], cache['per_cloud_qc']
    except FileNotFoundError:
        pass

    attached_ind = csd_stats[-1]
    n_attached   = len(attached_ind)
    factor       = np.float32(1.0 / float(nx * ny * nt))

    per_cloud_wq = np.zeros((n_attached, nz), dtype=np.float32)
    per_cloud_qc = np.zeros((n_attached, nz), dtype=np.float32)

    wq_path = f'{rundir}/scratch/hdf5/plume_all_wq.h5'
    qc_path = f'{rundir}/scratch/hdf5/plume_all_qc.h5'

    with h5py.File(wq_path, 'r') as fwq, h5py.File(qc_path, 'r') as fqc:
        ds_wq = fwq['wq']
        ds_qc = fqc['qc']
        NC    = ds_wq.shape[0]
        # Iterate over NC in contiguous chunks; filter to attached clouds inline
        # to keep HDF5 reads sequential (much faster than fancy indexing).
        j = 0  # index into per_cloud_* output arrays
        for i0 in range(0, NC, CHUNK):
            i1   = min(i0 + CHUNK, NC)
            # Which rows in this chunk belong to attached_ind?
            mask = (attached_ind >= i0) & (attached_ind < i1)
            if not mask.any():
                continue
            local_ids = attached_ind[mask] - i0   # offsets within chunk
            chunk_wq  = ds_wq[i0:i1]              # (CHUNK, NT, NZ)
            chunk_qc  = ds_qc[i0:i1]
            n_sel     = mask.sum()
            per_cloud_wq[j:j+n_sel] = chunk_wq[local_ids].sum(axis=1) * factor
            per_cloud_qc[j:j+n_sel] = chunk_qc[local_ids].sum(axis=1) * factor
            j += n_sel
            print(f'  {i1}/{NC}  ({j}/{n_attached} clouds)', end='\r', flush=True)
    print()

    np.savez(cache_path, per_cloud_wq=per_cloud_wq, per_cloud_qc=per_cloud_qc)
    print(f'Saved cache: {cache_path}')
    return per_cloud_wq, per_cloud_qc


def apply_bins(csd_stats, per_cloud_wq, per_cloud_qc, bins):
    """Bin the pre-computed per-cloud profiles.  Fast — no HDF5 I/O.

    Returns
    -------
    sum_wq, mean_wq, sum_qc, mean_qc : float32 arrays of shape (nbins, NZ)
    """
    clipped_mf = csd_stats[0]
    bin_ids    = np.digitize(np.log10(clipped_mf), bins)  # 1-indexed
    nbins      = len(bins) - 1

    for bnm in range(nbins):
        print(f'  bin {bnm+1}: {np.sum(bin_ids == bnm+1)}')

    sum_wq  = np.zeros((nbins, nz), dtype=np.float64)
    sum_qc  = np.zeros((nbins, nz), dtype=np.float64)
    cnt     = np.zeros(nbins, dtype=np.int64)

    for bnm in range(nbins):
        mask = bin_ids == (bnm + 1)
        if mask.any():
            sum_wq[bnm] = per_cloud_wq[mask].sum(axis=0)
            sum_qc[bnm] = per_cloud_qc[mask].sum(axis=0)
            cnt[bnm]    = mask.sum()

    c = cnt[:, np.newaxis]
    mean_wq = np.where(c > 0, sum_wq / np.maximum(c, 1), 0.0)
    mean_qc = np.where(c > 0, sum_qc / np.maximum(c, 1), 0.0)

    return (
        sum_wq.astype(np.float32),  mean_wq.astype(np.float32),
        sum_qc.astype(np.float32),  mean_qc.astype(np.float32),
    )

In [ ]:
# Step 1: load or compute per-cloud time-summed profiles (cached to disk)
cloud_profiles = []
for s, d in zip(stats, rundirs):
    print(f'\n=== {d} ===')
    cloud_profiles.append(compute_cloud_profiles(s, d))

In [ ]:
nbins = 15
bins = np.linspace(minmf, maxmf, nbins + 1)

# Step 2: bin the profiles (fast, re-run this cell to try different bins)
sum_wqs  = []
mean_wqs = []
sum_qcs  = []
mean_qcs = []
for s, (pwq, pqc) in zip(stats, cloud_profiles):
    sw, mw, sq, mq = apply_bins(s, pwq, pqc, bins)
    sum_wqs.append(sw)
    mean_wqs.append(mw)
    sum_qcs.append(sq)
    mean_qcs.append(mq)

In [ ]:
def compare_wq(sum_wqs, mean_wqs, casenames, casecolors, minmf, maxmf, plot_top=4.0, nbins=15):

    global zi
    bins = np.linspace(minmf, maxmf, nbins + 1)

    # 2-D difference panel
    fig, ax = plt.subplots(1, 1, figsize=(8, 12))

    sum_wq_diff = (sum_wqs[1] - sum_wqs[0]) * 1.0e3  # g/kg/m^2/s
    print(f"{casenames[1]}: max={sum_wq_diff.max()}, min={sum_wq_diff.min()}")

    levels = np.concatenate([np.arange(-14, -0.1, 2), [-0.1], [-0.01, 0.01], [0.1], np.arange(2, 14.1, 2)])
    cmap = copy.deepcopy(mpl.cm.bwr)
    norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
    cm = ax.pcolormesh(bins, zi * 1.0e-3, sum_wq_diff.T, norm=norm, cmap=cmap, shading='flat')
    ax.set_ylim(0, plot_top)
    ax.set_xlabel(r'$\log_{10}\left<M_b\right>$ (kg/s)', fontsize=18)
    ax.set_title(f"{casenames[1]} - CTL", fontsize=18)
    ax.set_ylabel('Height (km)', fontsize=18)

    fig.subplots_adjust(right=0.85)
    cbar_ax = fig.add_axes([0.87, ax.get_position().y0, 0.02, ax.get_position().height])
    cbar = plt.colorbar(cm, cax=cbar_ax)
    cbar.set_ticks(levels)
    cbar.ax.tick_params(labelsize=16)
    cbar.set_label(r"$\overline{w'q'}$ diff (10$^{-3}$ g kg$^{-1}$ m s$^{-1}$)", fontsize=18)
    plt.show()

    # Per-bin mean profiles
    bins_to_plot = [10, 11, 12, 13, 14]
    fig, axs = plt.subplots(1, len(bins_to_plot), figsize=(18, 8))
    axs = axs.flatten()
    fig.subplots_adjust(left=0.08, right=0.98, wspace=0.05)
    for iax, binno in enumerate(bins_to_plot):
        ax = axs[iax]
        ax.plot(mean_wqs[0][binno, :] * 1e3, z * 1.0e-3, color=casecolors[0], linewidth=2.5, label=casenames[0])
        for idx, m in enumerate(mean_wqs[1:]):
            ax.plot(m[binno, :] * 1e3, z * 1.0e-3, color=casecolors[idx+1], linewidth=2.5, label=casenames[idx+1])
        ax.set_ylim((0, plot_top))
        ax.grid(alpha=0.3)
        if iax != 0:
            ax.set_yticklabels([])
        if iax == 0:
            ax.set_ylabel('Height (km)')
        ax.set_title(f'Bin {binno+1}\n' + fr'{10**(bins[binno]):.0f}$\minus${10**(bins[binno+1]):.0f} kg/s', fontsize=14)
        if iax == 0:
            ax.legend(loc='upper right', fontsize=11, frameon=True, framealpha=0.9)
    fig.suptitle(r"Mean $\overline{w'q'}$ (10$^{-3}$ g kg$^{-1}$ m s$^{-1}$)", fontsize=24)
    plt.tight_layout()
    plt.show()

    # Per-bin sum profiles
    fig, axs = plt.subplots(1, len(bins_to_plot), figsize=(18, 8))
    axs = axs.flatten()
    fig.subplots_adjust(left=0.08, right=0.98, wspace=0.05)
    for iax, binno in enumerate(bins_to_plot):
        ax = axs[iax]
        ax.plot(sum_wqs[0][binno, :] * 1e3, z * 1.0e-3, color=casecolors[0], linewidth=2.5, label=casenames[0])
        for idx, m in enumerate(sum_wqs[1:]):
            ax.plot(m[binno, :] * 1e3, z * 1.0e-3, color=casecolors[idx+1], linewidth=2.5, label=casenames[idx+1])
        ax.set_ylim((0, plot_top))
        ax.grid(alpha=0.3)
        if iax != 0:
            ax.set_yticklabels([])
        if iax == 0:
            ax.set_ylabel('Height (km)')
        ax.set_title(f'Bin {binno+1}\n' + fr'{10**(bins[binno]):.0f}$\minus${10**(bins[binno+1]):.0f} kg/s', fontsize=14)
        if iax == 0:
            ax.legend(loc='upper right', fontsize=11, frameon=True, framealpha=0.9)
    fig.suptitle(r"Sum $\overline{w'q'}$ (10$^{-3}$ g kg$^{-1}$ m s$^{-1}$)", fontsize=24)
    plt.tight_layout()
    plt.show()

In [ ]:
def compare_qc(sum_qcs, mean_qcs, casenames, casecolors, minmf, maxmf, plot_top=4.0, nbins=15):

    global zi
    bins = np.linspace(minmf, maxmf, nbins + 1)

    # 2-D difference panel
    fig, ax = plt.subplots(1, 1, figsize=(8, 12))

    sum_qc_diff = (sum_qcs[1] - sum_qcs[0]) * 1.0e3
    print(f"{casenames[1]}: max={sum_qc_diff.max()}, min={sum_qc_diff.min()}")

    levels = np.concatenate([np.arange(-2.5, -0.01, 0.5), [-0.01, 0.01], np.arange(0.5, 2.51, 0.5)])
    cmap = copy.deepcopy(mpl.cm.bwr)
    norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
    cm = ax.pcolormesh(bins, zi * 1.0e-3, sum_qc_diff.T, norm=norm, cmap=cmap, shading='flat')
    ax.set_ylim(0, plot_top)
    ax.set_xlabel(r'$\log_{10}\left<M_b\right>$ (kg/s)', fontsize=18)
    ax.set_title(f"{casenames[1]} - CTL", fontsize=18)
    ax.set_ylabel('Height (km)', fontsize=18)

    fig.subplots_adjust(right=0.85)
    cbar_ax = fig.add_axes([0.87, ax.get_position().y0, 0.02, ax.get_position().height])
    cbar = plt.colorbar(cm, cax=cbar_ax)
    cbar.set_ticks(levels)
    cbar.ax.tick_params(labelsize=16)
    cbar.set_label(r"$q_c$ diff (10$^{-3}$ g kg$^{-1}$)", fontsize=18)
    plt.show()

    # Per-bin mean profiles
    bins_to_plot = [10, 11, 12, 13, 14]
    fig, axs = plt.subplots(1, len(bins_to_plot), figsize=(18, 8))
    axs = axs.flatten()
    fig.subplots_adjust(left=0.08, right=0.98, wspace=0.05)
    for iax, binno in enumerate(bins_to_plot):
        ax = axs[iax]
        ax.plot(mean_qcs[0][binno, :] * 1e3, z * 1.0e-3, color=casecolors[0], linewidth=2.5, label=casenames[0])
        for idx, m in enumerate(mean_qcs[1:]):
            ax.plot(m[binno, :] * 1e3, z * 1.0e-3, color=casecolors[idx+1], linewidth=2.5, label=casenames[idx+1])
        ax.set_ylim((0, plot_top))
        ax.grid(alpha=0.3)
        if iax != 0:
            ax.set_yticklabels([])
        if iax == 0:
            ax.set_ylabel('Height (km)')
        ax.set_title(f'Bin {binno+1}\n' + fr'{10**(bins[binno]):.0f}$\minus${10**(bins[binno+1]):.0f} kg/s', fontsize=14)
        if iax == 0:
            ax.legend(loc='upper right', fontsize=11, frameon=True, framealpha=0.9)
    fig.suptitle(r"Mean $q_c$ (10$^{-3}$ g kg$^{-1}$)", fontsize=24)
    plt.tight_layout()
    plt.show()

    # Per-bin sum profiles
    fig, axs = plt.subplots(1, len(bins_to_plot), figsize=(18, 8))
    axs = axs.flatten()
    fig.subplots_adjust(left=0.08, right=0.98, wspace=0.05)
    for iax, binno in enumerate(bins_to_plot):
        ax = axs[iax]
        ax.plot(sum_qcs[0][binno, :] * 1e3, z * 1.0e-3, color=casecolors[0], linewidth=2.5, label=casenames[0])
        for idx, m in enumerate(sum_qcs[1:]):
            ax.plot(m[binno, :] * 1e3, z * 1.0e-3, color=casecolors[idx+1], linewidth=2.5, label=casenames[idx+1])
        ax.set_ylim((0, plot_top))
        ax.grid(alpha=0.3)
        if iax != 0:
            ax.set_yticklabels([])
        if iax == 0:
            ax.set_ylabel('Height (km)')
        ax.set_title(f'Bin {binno+1}\n' + fr'{10**(bins[binno]):.0f}$\minus${10**(bins[binno+1]):.0f} kg/s', fontsize=14)
        if iax == 0:
            ax.legend(loc='upper right', fontsize=11, frameon=True, framealpha=0.9)
    fig.suptitle(r"Sum $q_c$ (10$^{-3}$ g kg$^{-1}$)", fontsize=24)
    plt.tight_layout()
    plt.show()

In [ ]:
compare_qc(sum_qcs, mean_qcs, casenames, casecolors, minmf, maxmf, nbins=15)

In [ ]:
compare_wq(sum_wqs, mean_wqs, casenames, casecolors, minmf, maxmf, nbins=15)